In [2]:
import pyarrow.parquet as pq
import os
import polars as pl

pl.Config.set_tbl_cols(30)          # Show up to 20 columns
pl.Config.set_tbl_rows(80)   
pl.Config.set_tbl_width_chars(500)  # Make the table wider in the console
pl.Config.set_fmt_str_lengths(50)   # Don't cut off long strings like stoch_key


# Define the file path
file_path = r'C:\Users\Owner\airflow-trading\data_lake\Opt_Session_20260427_002137_01\equity_partitioned\era_int=20220901\equity_era_int=20220901.parquet'

# Check if file exists to avoid errors
if os.path.exists(file_path):
    # Open the parquet file metadata
    parquet_file = pq.ParquetFile(file_path)
    
    # 1. Get the Schema
    schema = parquet_file.schema.to_arrow_schema()
    
    # 2. Get Total Row Count
    total_rows = parquet_file.metadata.num_rows
    
    print(f"--- File Statistics ---")
    print(f"Total Rows: {total_rows:,}")
    print(f"\n--- PyArrow Schema ---")
    print(schema)
else:
    print(f"Error: File not found at {file_path}")

--- File Statistics ---
Total Rows: 1,002

--- PyArrow Schema ---
regime_id: int32
signal_layer: int8
signal_scope_id: large_string
era_int: int64
side: int8
SL: float
TP: float
time_ns: int64
entry_idx: int64
exit_idx: int64
pnl_pct: float
equity: float


In [8]:
import polars as pl
import os

# 1. Point to the root directory
directory = r'C:\Users\Owner\airflow-trading\data_lake\Opt_Session_20260427_002137_01\equity_partitioned'

# 2. Use recursive glob pattern to find files in subdirectories
# **/*.parquet means "look in all subfolders for any parquet file"
path_pattern = os.path.join(directory, "**", "*.parquet")

print(f"Searching in: {path_pattern}")

# 3. Use scan_parquet for better performance with multiple files
# hive_partitioning=True automatically adds 'era_int' as a column in your data!
df = pl.scan_parquet(path_pattern, hive_partitioning=True).collect()

print(f"Total rows loaded: {len(df)}")
print(df.head())

Searching in: C:\Users\Owner\airflow-trading\data_lake\Opt_Session_20260427_002137_01\equity_partitioned\**\*.parquet
Total rows loaded: 11602
shape: (5, 12)
┌───────────┬──────────────┬─────────────────────────────────────┬──────────┬──────┬─────┬─────┬─────────────────────┬───────────┬──────────┬───────────┬────────────┐
│ regime_id ┆ signal_layer ┆ signal_scope_id                     ┆ era_int  ┆ side ┆ SL  ┆ TP  ┆ time_ns             ┆ entry_idx ┆ exit_idx ┆ pnl_pct   ┆ equity     │
│ ---       ┆ ---          ┆ ---                                 ┆ ---      ┆ ---  ┆ --- ┆ --- ┆ ---                 ┆ ---       ┆ ---      ┆ ---       ┆ ---        │
│ i32       ┆ i8           ┆ str                                 ┆ i64      ┆ i8   ┆ f32 ┆ f32 ┆ i64                 ┆ i64       ┆ i64      ┆ f32       ┆ f32        │
╞═══════════╪══════════════╪═════════════════════════════════════╪══════════╪══════╪═════╪═════╪═════════════════════╪═══════════╪══════════╪═══════════╪════════════╡
│ 97   

In [ ]:
# Returns the number of unique SL+TP pairs
unique_count = df.select(pl.col(["SL"])).n_unique()

print(f"Total Unique SL+TP Combinations: {unique_count}")

Total Unique SL+TP Combinations: 2
